In [0]:

from pyspark.sql import functions as F

CATALOG = "marketplace_catalog"
SCHEMA = "team6_marketplace"

def table(name):
    return spark.table(f"{CATALOG}.{SCHEMA}.{name}")

print("Data quality validation started.")


Data quality validation started.


In [0]:

tables_to_check = [
    "silver_customers",
    "silver_sellers",
    "silver_products",
    "silver_orders",
    "silver_order_items",
    "dim_customer",
    "dim_seller",
    "dim_product",
    "dim_date",
    "fact_order_items"
]

for name in tables_to_check:
    df = table(name)
    print(name, ":", df.count(), "rows")


silver_customers : 99441 rows
silver_sellers : 3095 rows
silver_products : 32951 rows
silver_orders : 99441 rows
silver_order_items : 112650 rows
dim_customer : 99441 rows
dim_seller : 3095 rows
dim_product : 32951 rows
dim_date : 634 rows
fact_order_items : 112650 rows


In [0]:

key_checks = {
    "silver_customers": ["customer_id"],
    "silver_sellers": ["seller_id"],
    "silver_products": ["product_id"],
    "silver_orders": ["order_id"],
    "silver_order_items": ["order_id", "order_item_id"],
    "dim_customer": ["customer_id"],
    "dim_seller": ["seller_id"],
    "dim_product": ["product_id"],
    "dim_date": ["date_key"],
    "fact_order_items": ["order_id", "order_item_id"]
}

for name, keys in key_checks.items():
    df = table(name)

    duplicate_count = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(name, "| Duplicate keys:", duplicate_count)


silver_customers | Duplicate keys: 0
silver_sellers | Duplicate keys: 0
silver_products | Duplicate keys: 0
silver_orders | Duplicate keys: 0
silver_order_items | Duplicate keys: 0
dim_customer | Duplicate keys: 0
dim_seller | Duplicate keys: 0
dim_product | Duplicate keys: 0
dim_date | Duplicate keys: 0
fact_order_items | Duplicate keys: 0


In [0]:

tables_for_null_check = [
    "silver_customers",
    "silver_orders",
    "silver_order_items",
    "silver_products",
    "silver_sellers",
    "fact_order_items"
]

for name in tables_for_null_check:
    df = table(name)

    print("\nTable:", name)

    null_counts = df.select([
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ])

    display(null_counts)



Table: silver_customers


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,load_timestamp,source_file_name
0,0,0,0,0,0,0



Table: silver_orders


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,load_timestamp,source_file_name
0,0,0,0,160,1783,2965,0,0,0



Table: silver_order_items


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,load_timestamp,source_file_name
0,0,0,0,0,0,0,0,0


... (2 outputs left out)


Table: silver_sellers


seller_id,seller_zip_code_prefix,seller_city,seller_state,load_timestamp,source_file_name
0,0,0,0,0,0



Table: fact_order_items


order_id,order_item_id,product_id,seller_id,customer_id,date_key,price,freight_value,order_status,order_delivered_customer_date,order_estimated_delivery_date,review_score
0,0,0,0,0,0,0,0,0,2454,0,1570


In [0]:

fact = table("fact_order_items")

orphan_checks = [
    ("customer_id", "dim_customer", "customer_id"),
    ("seller_id", "dim_seller", "seller_id"),
    ("product_id", "dim_product", "product_id"),
    ("date_key", "dim_date", "date_key")
]

for fact_key, dimension, dimension_key in orphan_checks:

    dimension_df = table(dimension)

    orphan_count = (
        fact.select(fact_key).distinct()
        .join(
            dimension_df.select(dimension_key).distinct(),
            fact[fact_key] == dimension_df[dimension_key],
            "left_anti"
        )
        .count()
    )

    print(fact_key, "| Orphan records:", orphan_count)


customer_id | Orphan records: 0
seller_id | Orphan records: 0
product_id | Orphan records: 0
date_key | Orphan records: 0


In [0]:

orders = table("silver_orders")

invalid_delivery_dates = (
    orders.filter(
        F.col("order_delivered_customer_date").isNotNull() &
        F.col("order_purchase_timestamp").isNotNull() &
        (
            F.col("order_delivered_customer_date") <
            F.col("order_purchase_timestamp")
        )
    )
)

print(
    "Invalid delivery date records:",
    invalid_delivery_dates.count()
)

display(invalid_delivery_dates)


Invalid delivery date records: 0


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,load_timestamp,source_file_name


In [0]:

quality_results = []

for name, keys in key_checks.items():

    df = table(name)

    row_count = df.count()

    duplicate_count = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_key_condition = None

    for key in keys:
        condition = F.col(key).isNull()
        null_key_condition = (
            condition if null_key_condition is None
            else null_key_condition | condition
        )

    null_key_count = df.filter(null_key_condition).count()

    quality_results.append((
        name,
        row_count,
        duplicate_count,
        null_key_count
    ))

quality_log = spark.createDataFrame(
    quality_results,
    [
        "table_name",
        "row_count",
        "duplicate_key_count",
        "null_key_count"
    ]
).withColumn(
    "validation_timestamp",
    F.current_timestamp()
)

quality_log.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.data_quality_log"
)

display(quality_log)


table_name,row_count,duplicate_key_count,null_key_count,validation_timestamp
silver_customers,99441,0,0,2026-11-07T05:33:16.402Z
silver_sellers,3095,0,0,2026-11-07T05:33:16.402Z
silver_products,32951,0,0,2026-11-07T05:33:16.402Z
silver_orders,99441,0,0,2026-11-07T05:33:16.402Z
silver_order_items,112650,0,0,2026-11-07T05:33:16.402Z
dim_customer,99441,0,0,2026-11-07T05:33:16.402Z
dim_seller,3095,0,0,2026-11-07T05:33:16.402Z
dim_product,32951,0,0,2026-11-07T05:33:16.402Z
dim_date,634,0,0,2026-11-07T05:33:16.402Z
fact_order_items,112650,0,0,2026-11-07T05:33:16.402Z
